# 04. Порівняння моделей на навчальних даних

Порівнюємо majority-class baseline, Logistic Regression, Decision Tree, Random Forest і CatBoost. Попередня обробка навчається окремо всередині кожного фолду. Відкладена тестова вибірка не використовується.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from src.data_split import load_dataset, split_dataset
from src.preprocessing import prepare_xy
from src.train import compare_models

data_path = project_root / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
dataset = load_dataset(data_path)
train, test = split_dataset(dataset)
del dataset, test
X, y = prepare_xy(train)
print(f'Навчальних записів: {len(X)}')

Навчальних записів: 5634


## Налаштування й результати

Використовуємо 5 стратифікованих фолдів із seed 42. Для кожної моделі вибираємо параметри за середнім F1 позитивного класу; поріг класифікації за замовчуванням не налаштовується. PR-AUC тут обчислюється як average precision. Значення після пошуку оптимістичні, бо параметри обиралися на тих самих фолдах; це не незалежна оцінка і не остаточний вибір моделі.

In [2]:
results = compare_models(X, y)
display(results.round(4))
for _, row in results.iterrows():
    print(f"{row['model']}: {row['best_params']}")

,model,best_params,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,Logistic Regression,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.5109,0.0120,0.8027,0.0187,0.6242,0.0116,0.8434,0.0102,0.6531,0.0309
1,CatBoost,"{'model__depth': 4, 'model__learning_rate': 0.05}",0.6701,0.0302,0.5157,0.0290,0.5821,0.0200,0.8465,0.0088,0.6605,0.0241
2,Random Forest,"{'model__max_depth': None, 'model__min_samples...",0.6699,0.0251,0.5037,0.0244,0.5744,0.0168,0.8432,0.0069,0.6506,0.0155
3,Decision Tree,"{'model__max_depth': 6, 'model__min_samples_le...",0.6244,0.0265,0.5211,0.0469,0.5666,0.0308,0.8283,0.0056,0.6230,0.0234
4,Baseline,{},0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.2654,0.0001


Logistic Regression: {'model__C': 1.0, 'model__class_weight': 'balanced'}
CatBoost: {'model__depth': 4, 'model__learning_rate': 0.05}
Random Forest: {'model__max_depth': None, 'model__min_samples_leaf': 5}
Decision Tree: {'model__max_depth': 6, 'model__min_samples_leaf': 30}
Baseline: {}


## Наступний етап

Визначити бізнес-метрику й поріг на навчальних/валідаційних даних, зокрема врахувати precision і recall, а не лише F1. Для менш упередженого порівняння налаштувань використати вкладену крос-валідацію або інший незалежний validation. Лише після всіх рішень оцінити фінальний pipeline на відкладеній тестовій вибірці. Вона не є повністю сліпою, оскільки первинний EDA охоплював весь набір.